## 멀티에이전트 구현

최종 프로젝트에서는 **에이전트 하나**가 데이터 조회 도구와 문서 검색 도구를 모두 가지고, 조회 → 검색 → 보고서 작성을 혼자 처리했습니다.

에이전트 하나로도 충분히 동작하지만, 맡기는 일이 많아지면 아래와 같은 문제가 생깁니다.

* **도구가 많아지면** : LLM이 어떤 도구를 써야 할지 헷갈리고, 잘못된 도구를 고르는 일이 늘어납니다.
* **시스템 프롬프트가 길어지면** : 데이터 조회 규칙, 문서 검색 규칙, 보고서 양식이 한곳에 섞여서 일부 규칙을 놓칩니다.
* **대화 기록이 길어지면** : 도구 결과(SQL 결과표, 검색된 문서 원문)가 계속 쌓여서 앞의 내용을 놓치고, 비용도 늘어납니다.

사람이 일하는 방식과 비슷하게 **역할을 나눠서** 이 문제를 줄일 수 있습니다. 파트장이 데이터 담당과 문서 담당에게 일을 나눠 주고, 받은 결과로 보고서를 쓰는 것처럼요. 이렇게 여러 에이전트가 역할을 나눠 협업하는 구조를 **멀티에이전트(Multi-Agent)** 라고 합니다.

이번 노트북에서 다룰 구조

| 패턴 | 한 줄 설명 |
|---|---|
| 1. 전문 에이전트 | 역할별로 도구와 프롬프트를 나눈 에이전트 만들기 |
| 2. 슈퍼바이저 (Supervisor) | 팀장 에이전트가 전문 에이전트를 **도구처럼** 불러서 일을 시킴 |
| 3. 라우터 (Router) | 질문을 분류해서 필요한 전문 에이전트에게만 보냄 |
| 4. 워크플로 (LangGraph) | 협업 순서를 사람이 그래프로 정해 둠 (작성 → 검토 → 재작성) |

사용하는 데이터는 최종 프로젝트와 같습니다. (`./db/fab_etch.db`, `./docs/`)

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

import os
from dotenv import load_dotenv, find_dotenv

# .env 파일로부터 api_key를 불러오는 코드
load_dotenv(find_dotenv())

model = ChatOpenAI(
    model="gpt-4o-mini",
    base_url=os.getenv("gpt_4o_mini_BASE_URL"),
    temperature=0,
)

도구는 `tools.py`에 미리 만들어 두었습니다. 최종 프로젝트에서 만든 도구와 같은 역할을 합니다.

| 도구 | 하는 일 |
|---|---|
| `chamber_summary` | 설비·챔버별 불량률과 공정 변수 비교 |
| `daily_trend` | 설비·챔버 하나의 날짜별 추이 |
| `run_sql` | 자유 SQL 조회 |
| `search_documents` | 설비 이력·인수인계 메모 검색 (이번 노트북에서는 임베딩 없이 키워드 검색으로 단순화) |

도구가 잘 동작하는지 먼저 확인해 봅시다.

In [ ]:
from tools import chamber_summary, daily_trend, run_sql, search_documents

print(chamber_summary.invoke({}))
print()
print(search_documents.invoke({"query": "03B 8월 16일 야간"})[:800])

### 1. 전문 에이전트 만들기

역할별로 에이전트를 나눕니다. 각 에이전트는 **자기 일에 필요한 도구만** 가지고, **자기 일에 대한 규칙만** 시스템 프롬프트에 적습니다.

* **데이터 분석 에이전트** : DB 도구만 사용. 이상이 있는 설비·챔버와 시작 날짜를 찾음
* **문서 조사 에이전트** : 문서 검색 도구만 사용. 요청받은 설비·챔버·날짜의 기록을 찾음

In [ ]:
data_agent = create_agent(
    model,
    tools=[chamber_summary, daily_trend, run_sql],
    system_prompt="""너는 반도체 Fab ETCH 공정 데이터 분석 담당이다.
1. chamber_summary로 설비·챔버별 불량률과 공정 변수를 비교해서, 다른 챔버와 다른 설비·챔버를 찾는다.
2. daily_trend로 그 설비·챔버의 날짜별 추이를 보고, 불량률과 공정 변수가 바뀌기 시작한 날짜를 찾는다.
3. 결과는 '설비·챔버 / 시작 날짜 / 바뀐 지표(바뀌기 전 → 후 값)' 형식으로 짧게 정리한다.
문서나 현장 기록은 볼 수 없으므로 원인을 추측하지 않는다.""",
)

doc_agent = create_agent(
    model,
    tools=[search_documents],
    system_prompt="""너는 ETCH 파트 설비 이력과 교대 인수인계 메모 조사 담당이다.
1. 요청받은 설비·챔버·날짜를 검색어에 넣어서 search_documents로 검색한다. 필요하면 검색어를 바꿔 여러 번 검색한다.
   (메모에서는 ETCH_03 CH_B를 '03B', ETCH_03을 '03호기'처럼 줄여 쓴다.)
2. 요청받은 설비·챔버의 기록만 골라서 시간 순서대로 정리한다. 날짜만 같고 설비·챔버가 다른 기록은 따로 표시한다.
3. 기록마다 이력번호(EH-2608-xxx) 또는 인수인계 날짜·교대(예: 08/16(일) 야간)를 적는다.
공정 수치는 볼 수 없으므로 문서에 적힌 내용만 쓴다.""",
)

각 전문 에이전트를 따로 실행해 봅시다. 역할이 좁기 때문에 결과도 해당 역할에 집중됩니다.

In [ ]:
response = data_agent.invoke(
    {"messages": [{"role": "user", "content": "8월에 불량률이 오른 설비·챔버와 시작 날짜를 찾아줘"}]}
)
print(response["messages"][-1].content)

In [ ]:
response = doc_agent.invoke(
    {"messages": [{"role": "user", "content": "ETCH_03 CH_B의 8월 16일~17일 기록을 찾아줘"}]}
)
print(response["messages"][-1].content)

### 2. 슈퍼바이저 패턴 : 에이전트를 도구로 쓰기

이제 두 에이전트에게 일을 나눠 줄 **팀장(슈퍼바이저) 에이전트**를 만듭니다.

방법은 간단합니다. 전문 에이전트를 실행하는 코드를 `@tool` 함수로 감싸면, 슈퍼바이저 입장에서는 **그냥 도구 하나**로 보입니다.

```
[사용자 질문] → 슈퍼바이저 ─┬─ ask_data_analyst(요청)       → 데이터 분석 에이전트 → 정리된 결과
                            └─ ask_document_researcher(요청) → 문서 조사 에이전트   → 정리된 결과
                → 보고서 작성
```

핵심 포인트

* 전문 에이전트는 **최종 정리 결과만** 돌려줍니다. SQL 결과표나 문서 원문은 슈퍼바이저의 대화 기록에 쌓이지 않아서, 슈퍼바이저는 판단에만 집중할 수 있습니다.
* 대신 슈퍼바이저가 전문 에이전트에게 보내는 **요청문(`request`)** 에 필요한 정보(설비·챔버·날짜)가 빠지면, 전문 에이전트는 엉뚱한 것을 찾습니다. 도구 설명(docstring)에 무엇을 넣어서 요청해야 하는지 적어 둡니다.

In [ ]:
@tool
def ask_data_analyst(request: str) -> str:
    """ETCH 공정 데이터(불량률, 압력, 온도, RF Power, 가스 유량) 분석 담당에게 일을 맡긴다.
    어느 설비·챔버에서 언제부터 수치가 바뀌었는지 알아야 할 때 사용한다.
    request에는 알고 싶은 내용과 기간을 구체적으로 적는다."""
    result = data_agent.invoke({"messages": [{"role": "user", "content": request}]})
    return result["messages"][-1].content


@tool
def ask_document_researcher(request: str) -> str:
    """설비 이력(PM, 점검, 알람조치, 변경점)과 교대 인수인계 메모 조사 담당에게 일을 맡긴다.
    특정 시점에 설비에서 무슨 일이 있었는지 알아야 할 때 사용한다.
    request에는 반드시 설비 ID, 챔버, 날짜(범위)를 넣는다."""
    result = doc_agent.invoke({"messages": [{"role": "user", "content": request}]})
    return result["messages"][-1].content


supervisor = create_agent(
    model,
    tools=[ask_data_analyst, ask_document_researcher],
    system_prompt="""너는 ETCH 파트 파트장을 돕는 보고서 작성 담당이다. 직접 데이터나 문서를 보지 않고, 담당자에게 일을 맡긴다.
불량률이나 공정 이상의 원인을 물으면 아래 순서로 진행한다.
1. ask_data_analyst에게 이상이 있는 설비·챔버와 시작 날짜를 찾게 한다.
2. 1에서 받은 설비·챔버와 시작 날짜, 그 전날을 넣어서 ask_document_researcher에게 그 시점의 기록을 찾게 한다.
3. 두 결과의 시각·설비·챔버를 맞춰 보고 연결 근거를 쓴다. 날짜만 같고 설비·챔버가 다른 기록은 원인으로 보지 않는다.
4. 확인되지 않은 내용은 '확인 필요'로 쓴다.
보고서 양식: 1) 현상 2) 관련 기록 3) 추정 원인과 근거 4) 원인에서 제외한 것 5) 확인 필요 사항""",
)

In [ ]:
response = supervisor.invoke(
    {"messages": [{"role": "user", "content": "8월 ETCH 불량률 올라간 거 원인 정리해서 보고서 초안 좀 써줘"}]}
)

# 슈퍼바이저가 누구에게 어떤 요청을 보냈는지 확인합니다.
for message in response["messages"]:
    for call in getattr(message, "tool_calls", None) or []:
        print(f"▶ {call['name']} : {call['args']['request']}")

print("\n" + response["messages"][-1].content)

슈퍼바이저의 대화 기록에 무엇이 쌓였는지 확인해 봅시다. 최종 프로젝트의 단일 에이전트와 비교하면 도구 결과가 훨씬 짧습니다. (SQL 결과표, 문서 원문 대신 담당자가 정리한 내용만 있습니다.)

In [ ]:
for message in response["messages"]:
    print(f"[{message.type}] 글자 수 {len(str(message.content))}")

**실습** : 최종 프로젝트의 질문 2, 3을 슈퍼바이저에게 해 보고, 어떤 담당자를 불렀는지 확인해 보세요.

* "8월 17일에 ETCH_02 레시피 변경은 관련 없어?"
* "03B 압력 경고는 언제부터 떴어?"

질문에 따라 담당자를 한 명만 부르거나, 두 명 모두 부르는지 살펴봅니다.

In [ ]:
question = "03B 압력 경고는 언제부터 떴어?"

response = supervisor.invoke({"messages": [{"role": "user", "content": question}]})

for message in response["messages"]:
    for call in getattr(message, "tool_calls", None) or []:
        print(f"▶ {call['name']} : {call['args']['request']}")
print("\n" + response["messages"][-1].content)

### 3. 라우터 패턴 : 질문을 분류해서 보내기

슈퍼바이저는 유연하지만, 담당자를 부를지 말지도 LLM이 판단하므로 매번 호출 횟수가 달라집니다.

질문 종류가 몇 가지로 정해져 있다면, **먼저 분류하고 필요한 담당자에게만 보내는** 라우터 구조가 더 단순합니다. `02_Langchain_파이프라인_심화.ipynb`에서 배운 **구조화된 출력**과 **병렬 실행**을 그대로 사용합니다.

```
[질문] → 분류 (data / document 중 필요한 것 선택)
       → 선택된 담당자들을 동시에 실행
       → 결과를 모아서 답변 작성
```

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel


class Route(BaseModel):
    targets: list[Literal["data", "document"]] = Field(
        description="질문에 답하는 데 필요한 담당자. "
                    "data: 불량률·압력 등 공정 수치가 필요할 때 / document: 설비 이력·인수인계 메모가 필요할 때. "
                    "둘 다 필요하면 둘 다 넣는다."
    )
    request: str = Field(description="담당자에게 보낼 요청문. 질문에 나온 설비·챔버·날짜를 빠뜨리지 않는다.")


classifier = (
    ChatPromptTemplate.from_messages([
        ("system", "ETCH 파트 엔지니어의 질문을 보고 필요한 담당자를 고른다."),
        ("user", "{question}"),
    ])
    | model.with_structured_output(Route)
)


# 담당자 이름 → 실행할 함수 (요청문을 받아 정리된 결과를 돌려줌)
WORKERS = {
    "data": lambda x: ask_data_analyst.invoke({"request": x["request"]}),
    "document": lambda x: ask_document_researcher.invoke({"request": x["request"]}),
}

answer_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "담당자들이 조사한 결과만 사용해서 질문에 답한다. 결과에 없는 내용은 '확인 필요'로 쓴다."),
        ("user", "질문: {question}\n\n조사 결과:\n{results}"),
    ])
    | model
    | StrOutputParser()
)


def run_router(question: str) -> str:
    route = classifier.invoke({"question": question})
    print(f"(선택된 담당자: {route.targets} / 요청문: {route.request})")

    # 선택된 담당자만 골라서 동시에 실행
    workers = RunnableParallel({name: RunnableLambda(WORKERS[name]) for name in route.targets})
    results = workers.invoke({"request": route.request})

    results_text = "\n\n".join(f"[{name}]\n{text}" for name, text in results.items())
    return answer_chain.invoke({"question": question, "results": results_text})


print(run_router("8월 ETCH_03 CH_B 불량률이 날짜별로 어떻게 바뀌었어?"))

In [ ]:
print(run_router("03B 압력 경고는 언제부터 떴어?"))

### 4. 워크플로 패턴 : LangGraph로 협업 순서 정하기

보고서처럼 **순서가 분명한 협업**은 순서를 LLM에게 맡기지 않고 사람이 정해 둘 수 있습니다. `create_agent()`도 내부적으로는 **LangGraph**라는 라이브러리로 만든 그래프입니다. LangGraph를 직접 사용하면 원하는 흐름을 그래프로 그릴 수 있습니다.

이번에는 아래 흐름을 만듭니다. 검토 담당이 통과시키지 않으면 작성 담당이 다시 씁니다. (무한 반복을 막기 위해 최대 2번까지만 다시 씁니다.)

```
START → 데이터 분석 → 문서 조사 → 보고서 작성 → 검토 ─(통과 또는 2회 재작성)→ END
                                     ↑            │
                                     └─(수정 요청)─┘
```

LangGraph의 기본 구성 요소

* **State(상태)** : 단계들이 함께 보고 쓰는 공용 메모장. 각 단계는 필요한 값을 읽고, 결과를 적습니다.
* **Node(노드)** : 한 단계. 상태를 받아서 바꿀 값만 딕셔너리로 돌려주는 함수입니다.
* **Edge(엣지)** : 다음에 어느 노드로 갈지 연결하는 선. 조건에 따라 갈라지는 엣지도 만들 수 있습니다.

In [ ]:
from typing import TypedDict


class ReportState(TypedDict):
    question: str        # 사용자 질문
    data_findings: str   # 데이터 분석 결과
    doc_findings: str    # 문서 조사 결과
    report: str          # 보고서 초안
    feedback: str        # 검토 의견
    approved: bool       # 검토 통과 여부
    revision: int        # 다시 쓴 횟수

각 단계(노드)를 함수로 만듭니다. 앞 단계의 결과를 다음 단계의 요청에 **코드로 확실하게** 넣어 준다는 점이 슈퍼바이저 패턴과 다릅니다.

In [ ]:
def analyze_data(state: ReportState) -> dict:
    result = data_agent.invoke({"messages": [{"role": "user", "content": state["question"]}]})
    return {"data_findings": result["messages"][-1].content}


def research_docs(state: ReportState) -> dict:
    # 데이터 분석 결과를 요청문에 그대로 넣어서, 문서 조사가 같은 설비·챔버·날짜를 찾도록 함
    request = (
        "아래 데이터 분석 결과에 나온 설비·챔버의, 시작 날짜와 그 전날 기록을 찾아줘.\n\n"
        + state["data_findings"]
    )
    result = doc_agent.invoke({"messages": [{"role": "user", "content": request}]})
    return {"doc_findings": result["messages"][-1].content}


writer_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "너는 ETCH 파트 보고서 작성 담당이다. 주어진 조사 결과만 사용한다.\n"
                   "데이터의 이상 시점과 문서 기록의 시각·설비·챔버를 맞춰 보고 연결 근거를 쓴다.\n"
                   "관련 기록에는 이력번호나 인수인계 날짜·교대를 함께 적는다.\n"
                   "보고서 양식: 1) 현상 2) 관련 기록 3) 추정 원인과 근거 4) 원인에서 제외한 것 5) 확인 필요 사항"),
        ("user", "질문: {question}\n\n[데이터 분석 결과]\n{data_findings}\n\n[문서 조사 결과]\n{doc_findings}\n\n"
                 "[이전 초안에 대한 검토 의견 (없으면 비어 있음)]\n{feedback}"),
    ])
    | model
    | StrOutputParser()
)


def write_report(state: ReportState) -> dict:
    report = writer_chain.invoke({**state, "feedback": state.get("feedback", "")})
    revision = state.get("revision", -1) + 1  # 처음 쓸 때 0, 다시 쓸 때마다 1씩 증가
    return {"report": report, "revision": revision}


class Review(BaseModel):
    approved: bool = Field(description="아래 기준을 모두 만족하면 true")
    feedback: str = Field(description="만족하지 못한 기준과 고칠 점. 통과면 빈 문자열")


reviewer_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "너는 ETCH 파트 보고서 검토 담당이다. 아래 기준으로 보고서를 검토한다.\n"
                   "1. 관련 기록마다 이력번호나 인수인계 날짜·교대가 적혀 있는가\n"
                   "2. 추정 원인의 설비·챔버가 데이터에서 이상이 나온 설비·챔버와 같은가\n"
                   "3. 조사 결과에 없는 내용을 사실처럼 쓰지 않았는가\n"
                   "4. 보고서 양식 5개 항목이 모두 있는가"),
        ("user", "[데이터 분석 결과]\n{data_findings}\n\n[문서 조사 결과]\n{doc_findings}\n\n[보고서]\n{report}"),
    ])
    | model.with_structured_output(Review)
)


def review_report(state: ReportState) -> dict:
    review = reviewer_chain.invoke(state)
    print(f"(검토 {state['revision'] + 1}회차: {'통과' if review.approved else '수정 요청'}) {review.feedback}")
    return {"approved": review.approved, "feedback": review.feedback}

이제 노드를 그래프에 등록하고 엣지로 연결합니다. `add_conditional_edges`는 함수의 반환값에 따라 다음 노드를 정합니다.

In [ ]:
from langgraph.graph import StateGraph, START, END

MAX_REVISION = 2


def after_review(state: ReportState) -> str:
    """검토 결과에 따라 다음 단계를 정함"""
    if state["approved"] or state["revision"] >= MAX_REVISION:
        return "end"
    return "rewrite"


builder = StateGraph(ReportState)

builder.add_node("analyze_data", analyze_data)
builder.add_node("research_docs", research_docs)
builder.add_node("write_report", write_report)
builder.add_node("review_report", review_report)

builder.add_edge(START, "analyze_data")
builder.add_edge("analyze_data", "research_docs")
builder.add_edge("research_docs", "write_report")
builder.add_edge("write_report", "review_report")
builder.add_conditional_edges("review_report", after_review, {"rewrite": "write_report", "end": END})

report_graph = builder.compile()

만든 그래프를 그림으로 확인해 봅시다. `02_Langchain_basic.ipynb`에서 에이전트 흐름을 그렸던 것과 같은 방법입니다.

In [ ]:
from IPython.display import Image, display

display(Image(report_graph.get_graph().draw_mermaid_png()))

In [ ]:
result = report_graph.invoke({"question": "8월 ETCH 불량률이 올라간 원인을 조사해줘"})

print("다시 쓴 횟수 :", result["revision"])
print(result["report"])

각 단계의 결과는 상태(`result`)에 모두 남아 있으므로, 어느 단계에서 문제가 생겼는지 따로 확인할 수 있습니다.

In [ ]:
print("[데이터 분석 결과]\n", result["data_findings"])
print("\n[문서 조사 결과]\n", result["doc_findings"])

### 정리 : 어떤 구조를 쓸까?

| 구조 | 순서를 정하는 주체 | 장점 | 주의할 점 | 어울리는 경우 |
|---|---|---|---|---|
| 단일 에이전트 | LLM | 만들기 가장 쉬움 | 도구·규칙이 많아지면 성능이 떨어짐 | 도구 몇 개로 끝나는 업무 |
| 슈퍼바이저 | LLM (팀장) | 질문에 따라 유연하게 담당자를 조합 | 요청문에 정보가 빠지면 담당자가 엉뚱한 것을 찾음, 호출 횟수가 일정하지 않음 | 질문 종류가 다양하고 조사 경로가 매번 다른 업무 |
| 라우터 | 분류 결과 + 코드 | 호출 횟수가 예측 가능, 담당자를 동시에 실행 | 담당자끼리 결과를 주고받아야 하는 일에는 부적합 | 질문 종류가 몇 가지로 정해진 문의 대응 |
| 워크플로 (LangGraph) | 사람 (그래프) | 순서가 항상 같아서 결과 확인이 쉬움, 검토·재시도 같은 반복을 넣기 쉬움 | 정해진 흐름 밖의 질문에는 대응이 어려움 | 보고서 작성처럼 절차가 정해진 업무 |

멀티에이전트는 LLM 호출 횟수가 늘어나서 **시간과 비용이 더 듭니다.** 단일 에이전트로 먼저 만들어 보고, 위의 문제(도구 선택 실수, 규칙 누락, 긴 대화 기록)가 실제로 생길 때 역할을 나누는 것을 권장합니다.

## 실습

**실습 1** : 슈퍼바이저에 **검토 담당**을 추가해 보세요.

* 보고서 초안을 받아서 인용 누락, 원인 설비 불일치 등을 지적하는 `review_agent`를 만들고 (도구 없이 `create_agent`로 만들어도 됩니다.)
* `ask_reviewer(report: str)` 도구로 감싸서 슈퍼바이저의 `tools`에 추가합니다.
* 슈퍼바이저 시스템 프롬프트에 "보고서를 쓴 뒤 검토를 받고, 지적 사항을 반영한다"는 규칙을 추가합니다.

In [ ]:
# review_agent = create_agent(...)

# @tool
# def ask_reviewer(report: str) -> str:
#     """..."""
#     ...

# supervisor_v2 = create_agent(...)

**실습 2** : 슈퍼바이저 패턴에서 `ask_document_researcher`의 docstring에서 `request에는 반드시 설비 ID, 챔버, 날짜(범위)를 넣는다.` 문장을 지우고 다시 실행해 보세요. 슈퍼바이저가 보내는 요청문과 보고서 결과가 어떻게 달라지는지 비교합니다.

**실습 3** : 워크플로 그래프에 **데이터 분석 결과를 사람이 확인하는 단계**를 넣으려면 어떻게 해야 할까요?

힌트 : `02_Langchain_basic.ipynb`의 `HumanInTheLoopMiddleware`처럼 LangGraph에도 `interrupt()`로 실행을 멈추고, `Command(resume=...)`로 이어서 실행하는 기능이 있습니다. (`checkpointer`가 필요합니다.) AI에게 예시 코드를 요청해서 적용해 보세요.